[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q3/00_orient_and_precommit.ipynb)

# R1-Q3 Week 1 — Orient and precommit

This notebook is where you commit, in writing, to the four decisions that shape the rest of R1-Q3:

- the attribution method you will use,
- the reference gene set you will compare against,
- the operational definition of "artifact-like" you will apply, and
- the data source and stress-class scope you will work with.

The file this notebook produces will be read by every notebook that follows, so the decisions you make here are the rails the rest of the analysis runs on.

By the end of this notebook you will have:

- A `precommit.json` file in your output directory recording all four decisions plus your reasoning for each, ready for the Week 2 classifier notebook to load.
- A short written statement (your own restatement of the question, plus the four precommit decisions and reasoning) ready as Week 1 EQ#1 input.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Orient to the question (read the page, restate it in your own words)

Open the R1-Q3 page in Notion (the question this notebook is built around) and read it end to end before you write anything below. While you read, hold onto four things:

- **The question itself** — what's being asked, in one sentence.
- **The Background** — why high test accuracy alone doesn't settle whether the classifier learned stress biology or learned dataset artifacts. The horse classifier in Lapuschkin et al. (2019) is the anchor example: a model that scored well on its test set was keying on a copyright watermark common in horse photos, not on horses themselves. SHAP (Lundberg & Lee, 2017) is the attribution tool the rest of this question's analysis runs on. Hakkak & Tohidfar (2026) is the reference list the top-attributed genes get compared against in Week 4.
- **The Prediction** — three labeled outcomes the page names. Strong Overlap means the top-attributed genes align with known stress-responsive genes. Mixed means some familiar genes show up alongside unfamiliar ones. Low Overlap means the top-attributed genes track technical metadata (batch, processing date, tissue) more than stress biology. The work of NB01 through NB03 is to determine which label fits.
- **The Considerations** — five bullets. Three of them turn into your four precommit decisions later in this notebook (Sections 3, 4, 5, and 6 — attribution method, reference gene set, the operational definition of "artifact-like," and data source plus stress-class scope; bullet 3 on the page splits into two precommits, attribution method and reference set). The other two bullets aren't written into `precommit.json`, but they shape what you do in NB01: bullet 1 (per-tissue classifiers vs tissue-as-covariate) is a tissue-handling decision you'll make in NB01 Section 3, and bullet 2 is the accuracy gate in NB01 Section 5 that decides whether running attribution is worth the work at all.

In the cell below, write a short paragraph (3–5 sentences) restating the question in your own words. A good restatement names: (a) what the classifier is trained on and what gets attributed — its predictions on a held-out AtGenExpress test split, with R1-Q3 self-contained on the AtGenExpress side (no Wang 2023); (b) what "relying on biology" versus "relying on artifacts" means concretely — overlap between top-attributed genes and known stress-responsive genes, plus non-correlation between attribution scores and technical metadata; and (c) the form the answer takes — one of three labeled verdicts (Strong, Mixed, Low Overlap), assigned by a precommitted comparison rule rather than by a pass/fail on some accuracy number.

This paragraph is for you. It isn't saved to `precommit.json`. Its job is to make sure you can argue the question is worth asking before you spend three weeks building the answer.

*(Write your restatement here. Replace this italicized text with your paragraph.)*

### 2) Inspect AtGenExpress experimental design (eight abiotic stresses × shoot/root × time-course, GEO range)

Before you run the cell below, open the **Rationale 1 datasets** page in Notion and read the AtGenExpress section. It covers the platform (ATH1 microarray, ~22k probes), the design (eight abiotic stresses × shoot/root × time-course), the two download paths (GEO 8-class vs TAIR/NASCArrays 9-class) that matter for Precommit 4 in Section 6, and the per-GSE structure of the GEO data — each stress treatment lives in its own GSE accession in the GSE5620–GSE5628 range.

The next cell loads the AtGenExpress sample metadata and prints summary counts — the same information in tabular form. Run it once you've read the dataset page.

In [ ]:
# Load AtGenExpress sample metadata and print:
#   - total sample count
#   - samples per stress condition (the per-class label distribution)
#   - samples per tissue (shoot vs root)
#   - samples per GSE accession
#     (compare to the per-stress counts above — each stress treatment
#      lives in its own GSE in the GEO 8-class scope)
#   - whether oxidative stress is present
#     (its presence distinguishes the TAIR/NASCArrays 9-class scope
#      from the GEO 8-class scope — this is what Precommit 4 resolves)
#   - the full list of metadata field names
#     (Precommit 3 will commit to testing batch, processing date,
#      and tissue — confirm all three are loadable here)

Five things to hold onto from the output. They all feed into decisions later in this notebook or in NB01:

- **Stress conditions present.** AtGenExpress covers eight or nine abiotic stresses depending on download path. Nine if oxidative is included (TAIR/NASCArrays), eight if not (GEO). This is what Precommit 4 (Section 6) resolves.
- **Shoot vs root counts.** AtGenExpress profiled both tissues, and tissue identity accounts for a larger share of expression variance than any single stress treatment. Two defensible strategies — train per-tissue classifiers, or model tissue as an explicit covariate so it can't act as a hidden shortcut. This is the decision in NB01 Section 3 (Considerations bullet 1 on the page).
- **Per-class sample sizes.** The classifier will train and test on these counts in NB01. If some classes have many fewer samples than others, attribution scores for those classes will be noisier in NB02, and the per-class hypergeometric test Precommit 3 commits to in Section 5 has correspondingly less statistical power for them.
- **Batch/GSE structure.** Each stress treatment lives in its own GSE accession in the GEO path (GSE5620–GSE5628). Batch and stress class are perfectly confounded by design: a classifier that learns "this sample is from GSE5621" can predict "drought" with 100% accuracy without learning anything about drought biology. This is the Clever Hans risk the Background flags, and it's the design-level reason the metadata variance-partitioning test in Precommit 3 (Section 5) exists.
- **Metadata fields available.** Confirm in the printed field list that batch/GSE, processing date, and tissue are all present. These are the three variables Precommit 3 commits to testing in Section 5 — if any of them isn't loadable, the precommit can't be implemented as written.

### 3) Precommit 1: attribution method (SHAP vs integrated gradients vs permutation importance)

This is Precommit 1, the first of four decisions you record before you run anything in NB01.

Once the classifier is trained in NB01, an attribution method assigns each gene a score showing how much that gene pushed each prediction toward or away from the predicted class. Different methods produce different gene rankings on the same model, and those rankings are what you'll compare against Hakkak & Tohidfar (2026) in Week 4. Considerations bullet 3 on the page names three methods:

- **SHAP** (Lundberg & Lee, 2017). Shapley-value-based attribution. For tree-based classifiers (random forest, gradient boosting), `shap.TreeExplainer` is the natural pairing and is computationally efficient. For neural networks, `shap.DeepExplainer` or `shap.KernelExplainer` apply. The most common attribution method in the genomics literature for this kind of question.
- **Integrated gradients** (Sundararajan et al., 2017). Gradient-based — integrates from a baseline input to the actual input. Requires a differentiable model, so it works on neural networks but not on tree-based classifiers without a surrogate. Library: `captum` (PyTorch) or `tf-explain` (TensorFlow).
- **Permutation importance**. Model-agnostic — randomize one feature at a time and measure the resulting drop in test accuracy. Library: `sklearn.inspection.permutation_importance`. Works for any model, but produces a single global score per gene (not per-prediction, not per-class), which is much coarser-grained than SHAP. Highly correlated features (which gene expression data has lots of) reduce reliability — when genes co-express, randomizing one can be effectively compensated for by its correlates.

All three are defensible. The choice has a real tie to the model architecture you'll train in NB01: tree-based models pair naturally with SHAP, neural networks work with all three, and permutation importance is the fallback that always works. If you haven't picked a model architecture yet, this precommit narrows your options for that downstream choice — better to face that constraint now than to find it in NB01 when an import fails.

NB02 Section 2 reads this decision to choose which attribution library to load and which call pattern to use. (See Considerations bullet 3 on the R1-Q3 page — this and Precommit 2 in Section 4 are the two halves of that bullet.)

*(Write 2–3 sentences here naming your choice and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Initialize the precommit dict. This is the first precommit section,
# so the dict gets created here; Sections 4, 5, and 6 will append to it,
# and Section 7 writes it to precommit.json.
#
# Record this section's decision under the "attribution_method" key:
#   - method: "SHAP", "integrated_gradients", or "permutation_importance"
#   - library_or_implementation: string naming the specific library
#     and class you'll call in NB02
#     (e.g., "shap.TreeExplainer",
#            "captum.attr.IntegratedGradients",
#            "sklearn.inspection.permutation_importance")
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above

### 4) Precommit 2: reference gene set (Hakkak & Tohidfar 2026 consensus DEGs, plus any others)

This is Precommit 2, the second of four decisions you record.

In Week 4 (NB03), your top-attributed genes get compared against a curated list of known stress-responsive genes — the overlap is the diagnostic that drives the Strong / Mixed / Low Overlap verdict. This precommit names the list (or lists) you'll compare against.

Two realistic choices for the **primary** reference:

- **Hakkak & Tohidfar (2026), the cross-stress consensus list.** A meta-analysis identifying genes differentially expressed across multiple plant stress types. The natural default — the page Background and Workflow row both name it explicitly, and the same list is used at the rationale level in R1-Q1's consensus comparison work.
- **An alternative curated source.** For example, PlantTFDB (a community-maintained transcription factor database) or one of the canonical stress-response gene lists compiled in Singhal (2025). Defensible if you want broader gene coverage than Hakkak, but less stress-specific — many PlantTFDB entries have nothing to do with abiotic stress, which makes the overlap test's interpretation noisier.

You can also **stack** additional sources on top of your primary. The JSON shape supports this (`primary_source` plus an `additional_sources` list, possibly empty). The overlap test in NB03 runs against the union of all loaded references — broader coverage means fewer false negatives (real stress genes the primary list missed), at the cost of mixing definitions of "stress-responsive" across sources. There's also a non-obvious downstream effect: a larger reference set raises the expected overlap by chance, so the same observed overlap is less surprising. Precommit 3 (Section 5) commits to the "above-chance" threshold for the hypergeometric test, and that threshold has to be applied against whatever reference-set size you commit to here.

Hakkak alone is the cleanest choice for a five-week project. Stacking is a defensible upgrade if you want to surface genes that Hakkak's consensus filtering excluded. Whatever you pick, lock it in here — swapping reference sets after seeing your attribution results turns the test into post-hoc curve-fitting.

NB03 Section 2 reads this decision to load the appropriate file(s) and run the per-class hypergeometric overlap test. (See Considerations bullet 3 on the R1-Q3 page — Precommit 1 and Precommit 2 are the two halves of that bullet.)

*(Write 2–3 sentences here naming your choice and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Append this section's decision to the precommit dict that Section 3
# initialized. Do not re-initialize the dict here.
#
# Record under the "reference_gene_set" key:
#   - primary_source: string, the main reference set name
#     (suggested values: "Hakkak_Tohidfar_2026", "PlantTFDB",
#      "Singhal_2025"; use a descriptive identifier if you have
#      another curated source in mind)
#   - additional_sources: list of strings (possibly empty),
#     each naming an additional curated source to stack on top
#     of the primary; the NB03 overlap test runs against the
#     union of all loaded references
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above

### 5) Precommit 3: operational definition of "artifact-like"

This is Precommit 3, the third of four decisions you record. It's structurally heavier than Precommits 1 and 2 because it commits to a *rule* with multiple parameters, not a single choice from a small menu.

The page Considerations bullet 4 frames the rule as a conjunctive test: flag a class's top-attributed genes as **artifact-like** when the overlap with the reference set from Precommit 2 is **no greater than chance** AND when the attribution scores are explained by technical metadata at a pre-specified threshold. That framing pins down the *shape* of the rule. Pinning down the *operational details* — which test, which threshold quantity, which threshold value, how per-class results roll up to one verdict — is what this precommit does.

The defaults below are reasonable settings that match standard practice in gene-enrichment and batch-confound literature. You can adopt them as-is, or adjust any of them with rationale in your reflection. Whatever you commit to here drives NB03 Section 5's verdict assignment.

**Granularity: per-class, with Bonferroni correction.** The classifier predicts one of 8 or 9 stress classes (the number depends on Precommit 4 in Section 6). The rule applies per class — each class gets its own overlap test and its own variance-partitioning test, and per-class verdicts roll up to the single Strong / Mixed / Low Overlap label. Per-class runs more tests, so Bonferroni correction divides the raw threshold by the number of classes to keep the family-wise false positive rate at the same level.

**Overlap clause: hypergeometric test.** For each class, build the 2×2 contingency table — (top-N attributed genes for this class) × (in your Precommit 2 reference set or not), against a background of all genes in the classifier's feature set — and run a hypergeometric test. The default threshold is `0.05 / n_classes` (e.g., `0.00625` for 8 classes, `0.00556` for 9). **The clause flags a class as artifact-like when the corrected p-value is *above* the threshold** — i.e., when the overlap is *not* significantly better than chance. The logic inverts from the usual "p < threshold means significant" reading because the rule defines artifact-like as the *absence* of a real overlap signal. Worth reading twice — getting this direction backwards turns every verdict on its head.

**Metadata clause: variance partitioning on top-attributed genes' expression.** For each class, take the top-N attributed genes and partition the variance in their expression across samples into components attributable to batch (GSE accession), processing date, tissue identity, and stress class. The default threshold is **30%** — if any of the three technical variables (batch, processing date, tissue) explains more than 30% of variance in the class's top-attributed genes, the metadata clause flags the class as artifact-like. The test asks the question Section 2 set up: do the genes the classifier most relied on for this class look more like batch markers than like stress markers?

**Combination: AND.** A class is flagged as artifact-like only when *both* clauses fail — overlap is not better than chance AND technical metadata explains an unreasonable share of variance. Either clause failing alone is suspicious but not damning; together they're converging evidence.

**Rollup to a single verdict: count-based.** Count the fraction of classes flagged as artifact-like (denominator excludes any class that can't be evaluated — see edge case below). The defaults are two cuts: fewer than **25%** of classes flagged → **Strong** (the classifier looks broadly biology-based); between 25% and 75% → **Mixed** (the picture is genuinely ambiguous); more than 75% → **Low Overlap** (most of what the classifier learned looks artifact-like). The three verdict labels match the page Prediction. The cuts are tunable.

**Edge case: unevaluable classes.** If a class's top-attributed genes don't overlap with your reference set at all (zero intersection), the hypergeometric test isn't meaningful for that class. The default is to **exclude it from the denominator** when counting flagged classes, with a note in the NB03 verdict explanation naming which classes were excluded and why. This prevents one biologically thin class from skewing the rollup.

Summary of the defaults:

| Piece | Default |
|---|---|
| Granularity | per-class |
| Overlap test | hypergeometric, Bonferroni-corrected |
| Overlap raw threshold | 0.05 (corrected: 0.05 / n_classes) |
| Metadata test | variance partitioning |
| Metadata variables | batch, processing date, tissue |
| Metadata threshold | 30% of variance |
| Clause combination | AND (both clauses must fail to flag the class) |
| Rollup rule | count-based, cuts at 25% and 75% of classes flagged |
| Verdict labels | Strong (<25%), Mixed (25–75%), Low Overlap (>75%) |
| Unevaluable classes | excluded from denominator with a note |

You commit to `n_classes` here, even though Precommit 4 (Section 6) is what nominally fixes it. The corrected overlap threshold depends on the number of classes the classifier trains on, so this section needs the number locked in before the threshold can be written down. Write down what you *intend* to commit to in Section 6 (probably 8 if you're going GEO, 9 if TAIR/NASCArrays). If your Section 6 choice ends up different, come back and update `n_classes` in Cell 5.D — the corrected threshold recomputes automatically from it.

NB03 Section 5 reads this entire dict to apply the rule and assign the verdict. (See Considerations bullet 4 on the R1-Q3 page for the long version.)

*(Write 3–5 sentences here. If you're adopting the defaults as-is, say so and explain why they fit your analysis. If you're adjusting any of them — different threshold, different metadata variables, different rollup cuts — name what you changed and the reasoning. The code cell below records the final numbers in machine-readable form — keep them consistent with what you wrote here.)*

In [ ]:
# Append this section's decision to the precommit dict that Section 3
# initialized. Do not re-initialize the dict here.
#
# The dict below pre-populates the defaults that match the prose in
# Cell 5.B. If your reflection adopted them as-is, you only need to
# replace the rationale string at the bottom. If you adjusted any of
# the numeric values or the metadata variables list, update them here
# before running the cell.

n_classes = 8  # ratify in Precommit 4 (Section 6); update here if Section 6 commits to 9

precommit["artifact_like_definition"] = {
    "n_classes": n_classes,
    "per_class": True,

    "overlap_clause": {
        "test": "hypergeometric",
        "threshold_quantity": "p_value",
        "multiple_testing_correction": "bonferroni",
        "raw_threshold": 0.05,
        "corrected_threshold": 0.05 / n_classes,
        "fails_when": "corrected_p_value > corrected_threshold",
    },

    "metadata_clause": {
        "test": "variance_partitioning",
        "variables": ["batch", "processing_date", "tissue"],
        "threshold_pct_variance_explained": 30,
        "fails_when": "any_variable_explains_more_than_threshold",
    },

    "combination": "AND",

    "rollup": {
        "rule": "count_based",
        "lower_cut": 0.25,
        "upper_cut": 0.75,
        "verdict_labels": {
            "below_lower_cut": "Strong",
            "between_cuts": "Mixed",
            "above_upper_cut": "Low_Overlap",
        },
        "unevaluable_class_handling": "exclude_from_denominator_with_note",
    },

    "rationale": "REPLACE WITH 1-3 sentences in your own words, consistent with the reflection cell above",
}

# Print confirmation.
print(f"Recorded artifact_like_definition with n_classes = {n_classes}.")
corrected = precommit["artifact_like_definition"]["overlap_clause"]["corrected_threshold"]
print(f"Corrected overlap p-value threshold = {corrected:.5f}.")

### 6) Precommit 4: data source + stress-class scope (8 vs 9 classes; GEO vs TAIR)

This is Precommit 4, the fourth and final decision you record before you run anything in NB01.

The AtGenExpress data lives in two places, and they don't contain the same set of stress conditions:

- **GEO range** — eight abiotic stresses (the eight you saw in Section 2's output, which does *not* include oxidative).
- **TAIR/NASCArrays** — all nine stresses, adding oxidative.

Either is defensible. The question is what scope claim you want to make in your final paper and presentation. Going with GEO means your attribution analysis covers eight stresses' worth of top-attributed gene rankings, and any "my classifier learned stress biology" or "my classifier learned dataset artifacts" claim only generalizes to those eight. Going with TAIR/NASCArrays adds oxidative — one more class, one more per-class hypergeometric test in NB03, one more chance for the rollup to land at Mixed instead of Strong (or Strong instead of Mixed). Pick the download path that matches the scope claim you want to make, and write the reasoning down here.

A second consequence to think through: this section closes the loop with Section 5. Precommit 3 uses `n_classes` to compute the Bonferroni-corrected overlap threshold (`0.05 / n_classes`) and to set the denominator for the rollup test. If your choice here differs from what you wrote in Cell 5.D, go back and update Section 5's `n_classes` value — the corrected threshold recomputes automatically when you re-run that cell.

NB01 Section 2 reads this decision to choose which AtGenExpress download path to load. NB02 Section 2 reads it to know how many per-class attribution lists to produce. NB03 Section 5 reads it via Precommit 3 to apply the artifact-like rule across the right number of classes. (See Considerations bullet 5 on the R1-Q3 page for the long version.)

*(Write 2–3 sentences here naming your choice and the reasoning behind it. The code cell below records the same decision in machine-readable form — keep them consistent.)*

In [ ]:
# Append this section's decision to the precommit dict that Section 3
# initialized. Do not re-initialize the dict here.
#
# Record under the "data_source_and_scope" key:
#   - source: "GEO" or "TAIR_NASCArrays"
#   - n_stress_classes: 8 (GEO) or 9 (TAIR_NASCArrays)
#   - rationale: 1-2 sentences in the student's own words,
#     consistent with the reflection cell above
#
# If your choice here differs from the n_classes you wrote in
# Cell 5.D, also update Cell 5.D and re-run it so the corrected
# overlap threshold matches.

### 7) Closeout: write precommit.json, submit EQ#1

You've reached the end of NB00. Four precommit decisions are now sitting in memory (Section 3 initialized the dict; Sections 4, 5, and 6 each appended to it). What's left is to write the dict to disk, verify the file looks right, and submit EQ#1.

The cell below writes `precommit.json` to your R1-Q3 output directory (`OUTPUT_DIR`, set up in Cell 2). Run it, then open the file and read it back. Not because the write might fail silently — because EQ#1 is your record of what you committed to *before* you ran any analysis. NB01, NB02, and NB03 all open this file in their first body section. If a field is wrong here, every downstream notebook reads the wrong field.

One specific consistency to check when you read the file back: `artifact_like_definition.n_classes` (Precommit 3, from Cell 5.D) and `data_source_and_scope.n_stress_classes` (Precommit 4, from Cell 6.D) should be the same number. If they don't match, go back to whichever cell you adjusted last, update the other to match, and re-run it before finalizing the file.

In [ ]:
# Write the precommit dict to OUTPUT_DIR / "precommit.json".
# Use json.dump with indent=2 for human readability.
#
# Then read the file back from disk and pretty-print it,
# so you can visually confirm the four decisions and their
# rationales before submitting.
#
# Expected top-level keys (set in Sections 3, 4, 5, 6):
#   - attribution_method
#   - reference_gene_set
#   - artifact_like_definition
#   - data_source_and_scope

Your EQ#1 deliverable has two parts: the `precommit.json` file you just wrote, and the four short reflection paragraphs you wrote in the markdown cells of Sections 3, 4, 5, and 6 (those live inside the notebook itself). Submit per your program's EQ#1 procedure — see the iRI Mentor 2026 onboarding SOP if you're unsure where to upload.

Once EQ#1 is submitted, NB01 picks up next week. Its first body section is a defensive load of this same `precommit.json` — every decision you locked in here drives a branch downstream.